# 01d — Merge All Datasets

Reads all preprocessed datasets from Drive, merges into one, splits 80/20 train/val, and saves as a single zip to Drive.

Run this **after** 01a, 01b, 01c. Then run 02_train.ipynb which unzips the merged dataset.

| | |
|---|---|
| **Input** | Drive: datasets/qasim21, datasets/nexdata, datasets/roboflow |
| **Output** | Drive: merged_dataset.zip |

In [ ]:
import os, shutil, glob, random, yaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AI_TRAINING/GreenVision'
DATASETS_ROOT = os.path.join(DRIVE_ROOT, 'datasets')
MERGED_DIR = '/content/dataset/merged'
ZIP_PATH = os.path.join(DRIVE_ROOT, 'merged_dataset.zip')

## 1. Scan Datasets on Drive

In [ ]:
DATASETS = []
if os.path.exists(DATASETS_ROOT):
    for entry in sorted(os.listdir(DATASETS_ROOT)):
        ds_path = os.path.join(DATASETS_ROOT, entry)
        if os.path.isdir(ds_path) and os.path.isdir(os.path.join(ds_path, 'images')):
            DATASETS.append(entry)

print(f'Found {len(DATASETS)} datasets in {DATASETS_ROOT}:')

total_imgs = 0
total_lbls = 0
for ds_name in DATASETS:
    ds_path = os.path.join(DATASETS_ROOT, ds_name)
    imgs = glob.glob(os.path.join(ds_path, 'images', '*.*'))
    lbls = glob.glob(os.path.join(ds_path, 'labels', '*.txt'))
    print(f'  {ds_name}: {len(imgs)} images, {len(lbls)} labels')
    total_imgs += len(imgs)
    total_lbls += len(lbls)

print(f'\nTotal: {total_imgs} images, {total_lbls} labels')

if total_imgs == 0:
    raise SystemExit('No datasets found! Run 01a/01b/01c first.')

## 2. Merge & Split 80/20

In [ ]:
# Clean merged directory
if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
os.makedirs(f'{MERGED_DIR}/images/train', exist_ok=True)
os.makedirs(f'{MERGED_DIR}/images/val', exist_ok=True)
os.makedirs(f'{MERGED_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{MERGED_DIR}/labels/val', exist_ok=True)

# Collect all image-label pairs
random.seed(42)
all_pairs = []

for ds_name in DATASETS:
    ds_path = os.path.join(DATASETS_ROOT, ds_name)
    imgs = glob.glob(os.path.join(ds_path, 'images', '*.jpg')) + \
           glob.glob(os.path.join(ds_path, 'images', '*.png'))
    for img_path in imgs:
        base = os.path.splitext(os.path.basename(img_path))[0]
        ext = os.path.splitext(img_path)[1]
        lbl_path = os.path.join(ds_path, 'labels', base + '.txt')
        if os.path.exists(lbl_path):
            all_pairs.append((img_path, lbl_path, ext, ds_name))

random.shuffle(all_pairs)
val_count = int(len(all_pairs) * 0.2)

print(f'Total pairs: {len(all_pairs)}')
print(f'Train: {len(all_pairs) - val_count}, Val: {val_count}')

# Copy to merged directory with dataset prefix
for i, (img_path, lbl_path, ext, src_ds) in enumerate(all_pairs):
    base = os.path.splitext(os.path.basename(img_path))[0]
    name = f'{src_ds}_{base}'
    split = 'val' if i < val_count else 'train'

    dst_img = f'{MERGED_DIR}/images/{split}/{name}{ext}'
    dst_lbl = f'{MERGED_DIR}/labels/{split}/{name}.txt'

    if os.path.exists(dst_img):
        name = f'{name}_{random.randint(1000,9999)}'
        dst_img = f'{MERGED_DIR}/images/{split}/{name}{ext}'
        dst_lbl = f'{MERGED_DIR}/labels/{split}/{name}.txt'

    shutil.copy2(img_path, dst_img)
    shutil.copy2(lbl_path, dst_lbl)

train_imgs = len(glob.glob(f'{MERGED_DIR}/images/train/*.*'))
val_imgs = len(glob.glob(f'{MERGED_DIR}/images/val/*.*'))
train_lbls = len(glob.glob(f'{MERGED_DIR}/labels/train/*.txt'))
val_lbls = len(glob.glob(f'{MERGED_DIR}/labels/val/*.txt'))

print(f'\nMerged dataset:')
print(f'  Train: {train_imgs} images, {train_lbls} labels')
print(f'  Val:   {val_imgs} images, {val_lbls} labels')

## 3. Write data.yaml

In [ ]:
config = {
    'path'  : '/content/dataset/merged',
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : 1,
    'names' : {0: 'person'},
}

yaml_path = os.path.join(MERGED_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('data.yaml written:')
print(open(yaml_path).read())

## 4. Zip & Save to Drive

In [ ]:
import subprocess

# Remove old zip if exists
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# Zip the merged directory
print('Zipping merged dataset...')
result = subprocess.run(
    ['zip', '-r', '-q', ZIP_PATH, 'merged'],
    cwd='/content/dataset',
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f'Zip error: {result.stderr}')
else:
    size_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
    print(f'Saved: {ZIP_PATH}')
    print(f'Size: {size_mb:.1f} MB')

# Verify zip contents
print('\nVerifying zip...')
result = subprocess.run(['unzip', '-l', ZIP_PATH], capture_output=True, text=True)
lines = result.stdout.strip().split('\n')
img_count = sum(1 for l in lines if '/images/' in l and l.endswith(('.jpg', '.png')))
lbl_count = sum(1 for l in lines if '/labels/' in l and l.endswith('.txt'))
print(f'  Images in zip: {img_count}')
print(f'  Labels in zip: {lbl_count}')

if img_count != train_imgs + val_imgs:
    print(f'WARNING: Expected {train_imgs + val_imgs} images, got {img_count}')
if lbl_count != train_lbls + val_lbls:
    print(f'WARNING: Expected {train_lbls + val_lbls} labels, got {lbl_count}')

---
## Done!

Merged dataset saved as zip to:
```
/content/drive/MyDrive/AI_TRAINING/GreenVision/merged_dataset.zip
```

Next: **02_train.ipynb** (unzips and trains)